# fMRI Connectomics & Machine Learning Pipeline

Today, you will build a complete functional Magnetic Resonance Imaging (fMRI) pipeline: from raw-ish BOLD signals to brain connectivity graphs, and finally to a machine learning classifier.

For this exercise you only need a normal Python runtime type, as we won't need GPUs.

Let's start by installing and importing the right packages

In [ ]:
# Installing nilearn package
!pip install nilearn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from networkx.algorithms import community

from nilearn import datasets, plotting
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

## Exercise 1: Loading & Visualising Brain Data

We will use the **Schaefer 2018 atlas** (100 regions) to parcellate the brain. The Schaefer atlas is available in multiple "resolutions". You can ask for 100, 200, 400, or up to 1000 ROIs. You can also ask for them to be mapped to either 7 or 17 networks. For the pipeline that you will develop, the ML model will use the connections between the 100 regions. But later, when you visualise the most important edges, knowing the rough 7 networks can be helpful for interpretation. You can ignore this detail for the rest of the exercise, and use this information to just know how to call the right functions.

Then, we will load 50 subjects from the **Development fMRI dataset** (predicting Child vs. Adult). This dataset has 155 subjects available, but given the time it takes, we will start with only 50 - you'll see that despite this small number, Colab still takes quite a while in some steps, showing once again the challenges of typical biomedical data.


**Pointers:**
1. Fetch the Schaefer 2018 atlas with 100 ROIs and 7 networks.
2. Plot the atlas over a standard MRI template.
3. Fetch the Development dataset (`n_subjects=50`).
4. Extract the target labels (1 for Child, 0 for Adult).

In [ ]:
# atlas = datasets.fetch_...
# plotting.plot_roi(...)
# dev_data = datasets.fetch_development_fmri(...)

# Exercise 2: Extract Time-Series

Using Nilearn's `NiftiLabelsMasker`, extract the mean time-series for each of the 100 regions for all subjects. To be clear and to avoid potential data leakage, you need to calculate the mean for each patient independently.

*(This might take a few minutes on Colab)*

**Pointers:**
- Remember to pass the `confounds` to clean the signal of noise!
- Ensure the extracted signals are standardised(z-scored) per sample. This ensures the time-series are on a comparable scale across different subjects. Explore the parameters of `NiftiLabelsMasker` to find how to apply this scaling.

In [ ]:
# masker = NiftiLabelsMasker(...)
# time_series_all = []
# for img, conf in zip(fmri_filenames, confounds):
#     time_series = masker.fit_transform(...)
#     time_series_all.append(time_series)

# Exercise 3: Computing Functional Connectivity

Compute standard **correlation** and **partial correlation** matrices from the previous timeseries.

**Pointers:**
1. Use `ConnectivityMeasure` to fit and transform the time series.
2. Plot the correlation matrix of the first subject and compare it to the partial correlation one.
3. Apply a **proportional threshold** (e.g., top 5% edges) and visualize the brain connectome using `plotting.plot_connectome`.

In [ ]:
# conn_corr = ConnectivityMeasure(kind=...)
# matrices_corr = ...
# ...
# plotting.plot_matrix(...)
# plotting.plot_connectome(...)


## Exercise 4: Graph Analysis

Let's treat the brain as a graph $G = (V, E)$ to extract topological features.

**Pointers:**
1. Take the correlation matrix of the first subject.
2. Apply an **absolute threshold** (e.g., keep edges > 0.4) to create a binary adjacency matrix.
3. Convert this into a `NetworkX` graph.
4. Compute:
   - Average Clustering Coefficient
   - Top 3 nodes by Degree (hubs)
   - Top 3 nodes by Betweenness Centrality
   - Graph Modularity (using `nx.community.greedy_modularity_communities`)

In [ ]:
# ...
# adj_matrix = ... # Threshold the matrix
# ...
# G = nx.from_numpy_array(adj_matrix)
# ...
# clustering = nx.average_clustering(G)
# ...
# communities = community.greedy_modularity_communities(G)
# ...

# Exercise 5: Predicting Age Group (Child vs Adult) & Data Leakage

A common mistake in neuroimaging ML is **data leakage**.
Two common sources of leakage are:
1. **Feature Selection:** Selecting top features on the entire dataset before splitting into train/test folds.
2. **Tangent Embedding:** `ConnectivityMeasure(kind='tangent')` calculates a **group mean covariance matrix**. If you apply `.fit_transform()` on the entire dataset, information from the test subjects leaks into the spatial embedding of the training subjects!

In this exercise, compare two approaches using a Linear SVM and 5-fold Stratified cross-validation.
1. **The Leaky Way:** Apply `ConnectivityMeasure(kind='tangent', vectorize=True)` and `SelectKBest` on the *entire* list of time-series. Then run cross-validation.
2. **The Correct Pipeline Way:** Create an `sklearn.pipeline.Pipeline` containing the `ConnectivityMeasure`, `StandardScaler`, `SelectKBest`, and the `SVC`. This guarantees the Tangent mean and feature selection are **only** fit on the training data of each fold!

*Calculate AUROC and AUPRC for both to quantify performance*

In [ ]:
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# model = SVC(probability=True, kernel='linear')
# y = labels

# # 1. Leaky Way
# X_leaky_tangent = ConnectivityMeasure(...).fit_transform(time_series_all)
# X_leaky_selected = SelectKBest(...).fit_transform(X_leaky_tangent, y)
# ... cross_val_score ...

# # 2. Strict Pipeline
# strict_pipe = Pipeline([
#     ('conn', ConnectivityMeasure(...)),
#     ('scaler', StandardScaler()),
#     ('selector', SelectKBest(...)),
#     ('svm', model)
# ])
# ... cross_val_score ...

# Exercise 6: Visualising the Model's Brain
Machine Learning in neuroscience is mostly about interpretation. Let's find out *which* brain connections distinguish children from adults.

**Pointers:**
1. Fit your **strict pipeline** on the *entire* dataset (this is fine now, because we are no longer evaluating, just inspecting the final model for deployment).
2. Extract the weights (coefficients) from the linear SVM.
3. Because we used `SelectKBest`, map these weights back to the full feature space.
4. Use `conn.inverse_transform()` to turn the 1D feature vector back into a 100x100 symmetric matrix.
5. Plot the top 1% of discriminative edges.

In [ ]:
# strict_pipe.fit(time_series_all, y)
# svm_weights = strict_pipe.named_steps['svm'].coef_[0]
# ... map weights back ...
# plotting.plot_connectome(...)

# Exercise 7: Parameter Exploration
You have built a rigorous, leakage-free pipeline! If you got here, try modifying parameters to see how they affect model performance and interpretability.

- **Classifier Choice:** Swap the `SVC` for a `RandomForestClassifier`. Does non-linear modelling improve accuracy? Can you extract `feature_importances_` instead of `coef_`?
- **Feature Selection:** Change `SelectKBest` from `k=100` to `k=500` or `k='all'`.
- **Correlation vs. others:** Is there a difference in performance between using full correlations or tangent space?

